# Forward Kinematics

This notebook computes forward kinematics from the DH table and verifies the same joint command in CoppeliaSim.

In [1]:
import numpy as np
import time
from coppeliasim_zmqremoteapi_client import RemoteAPIClient

def FK(theta, alpha, a, d):
    cth = np.cos(theta)
    sth = np.sin(theta)
    cal = np.cos(alpha)
    sal = np.sin(alpha)

    return np.array([
        [cth, -sth * cal,  sth * sal, a * cth],
        [sth,  cth * cal, -cth * sal, a * sth],
        [0.0,  sal,        cal,       d],
        [0.0,  0.0,        0.0,       1.0]
    ])

def get_ee_matrix(sim, ee_handle):
    T = np.array(sim.getObjectMatrix(ee_handle, -1)).reshape(3, 4)
    return np.vstack((T, [0.0, 0.0, 0.0, 1.0]))

# Example joint angles from the paper's Franka Panda configuration in radians
q = np.array([1.157, -1.066, -0.155, -2.239, -1.841, 1.003, 0.469])

dh_params = [
    (q[0],  0.0,        0.0,     0.333),
    (q[1], -np.pi / 2,  0.0,     0.0),
    (q[2],  np.pi / 2,  0.0,     0.316),
    (q[3],  np.pi / 2,  0.0825,  0.0),
    (q[4], -np.pi / 2, -0.0825,  0.384),
    (q[5],  np.pi / 2,  0.0,     0.0),
    (q[6],  0.0,        0.088,   0.0),
]

T0_7 = np.eye(4)
for i, params in enumerate(dh_params, start=1):
    T0_7 = T0_7 @ FK(*params)
    print(f'T0_{i} =')
    print(np.round(T0_7, 4))
    print()

print('=' * 50)
print('ANALYTICAL END-EFFECTOR TRANSFORMATION T0_7')
print('=' * 50)
print(np.round(T0_7, 4))
print('\nAnalytical end-effector position [x, y, z] in meters:')
print(np.round(T0_7[:3, 3], 4))

# CoppeliaSim setup
client = RemoteAPIClient()
sim = client.getObject('sim')
client.setStepping(False)
sim.startSimulation()
print('\nConnected to CoppeliaSim and simulation started.')

joint_names = [f'/Franka/joint_{i + 1}' for i in range(7)]
joint_handles = [sim.getObjectHandle(name) for name in joint_names]
ee_handle = sim.getObjectHandle('/Franka/EE_Dummy')

for i in range(7):
    sim.setJointTargetPosition(joint_handles[i], float(q[i]))

time.sleep(1.0)
T_ee_sim = get_ee_matrix(sim, ee_handle)

print('\n' + '=' * 50)
print('COPPELIASIM END-EFFECTOR TRANSFORMATION')
print('=' * 50)
print(np.round(T_ee_sim, 4))
print('\nSimulated end-effector position [x, y, z] in meters:')
print(np.round(T_ee_sim[:3, 3], 4))

position_error = np.linalg.norm(T0_7[:3, 3] - T_ee_sim[:3, 3])
rotation_error = np.linalg.norm(T0_7[:3, :3] - T_ee_sim[:3, :3])

print('\nPosition error:', round(float(position_error), 6))
print('Rotation error:', round(float(rotation_error), 6))


T0_1 =
[[ 0.4021 -0.9156  0.      0.    ]
 [ 0.9156  0.4021  0.      0.    ]
 [ 0.      0.      1.      0.333 ]
 [ 0.      0.      0.      1.    ]]

T0_2 =
[[ 0.9959 -0.     -0.0909  0.    ]
 [ 0.0909  0.      0.9959  0.    ]
 [ 0.     -1.      0.      0.333 ]
 [ 0.      0.      0.      1.    ]]

T0_3 =
[[ 0.9839 -0.0909 -0.1537 -0.0287]
 [ 0.0898  0.9959 -0.014   0.3147]
 [ 0.1544  0.      0.988   0.333 ]
 [ 0.      0.      0.      1.    ]]

T0_4 =
[[-0.5383 -0.1537 -0.8286 -0.0731]
 [-0.8373 -0.014   0.5465  0.2456]
 [-0.0957  0.988  -0.1212  0.3251]
 [ 0.      0.      0.      1.    ]]

T0_5 =
[[ 0.2918  0.8286 -0.4777 -0.4154]
 [ 0.237  -0.5465 -0.8032  0.4359]
 [-0.9266  0.1212 -0.3559  0.355 ]
 [ 0.      0.      0.      1.    ]]

T0_6 =
[[ 0.8555 -0.4777 -0.1996 -0.4154]
 [-0.3333 -0.8032  0.4937  0.4359]
 [-0.3962 -0.3559 -0.8464  0.355 ]
 [ 0.      0.      0.      1.    ]]

T0_7 =
[[ 0.5472 -0.8128 -0.1996 -0.3672]
 [-0.6604 -0.5658  0.4937  0.3778]
 [-0.5142 -0.1384 -0.8464  0.